## **Carregar bibliotecas**

In [1]:
# Load the used classes
from modules_otft._imports import *
from modules_otft._read_data import ReadData
from modules_otft._model import TFTModel
from modules_otft._grafics import  TFTGraphicsPlot
from modules_otft._menu import TFTMenu

# from modules_otft._global_vars import *
from modules_otft._config import *
from modules_otft._utils import *
from modules_otft._display import display_settings

## **Gerar dados sintéticos**   

In [2]:
from modules_otft._generate_synthetic_data import SyntheticFromSettings

plot = TFTGraphicsPlot()
menu = TFTMenu()
read = ReadData()



In [8]:
# Read and show the path file Json
settings = enter_with_json_file()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


___________________________________________________________________________________ SETTINGS PRESENT IN THE JSON FILE:___________________________________________________________________________________

| path: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/datas/Org1/tipo p
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| experimental_data_scale_transfer: A
---------------------------------------------------------------------------

In [9]:
ld_voltages = get_load_voltages(settings)

type_curve_plot = get_type_plot(settings)

shift_list = calculate_shift_list(settings)

list_tension_shift = get_shift_list(read, settings)

path_voltages = read.read_files_experimental(settings['path'], list_tension_shift)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)


In [11]:
# Caso o Shift seja aplicado 
path_voltages, new_values_tension, new_list_tension = filter_and_load_files(read, settings, path_voltages, list_tension_shift)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)
list_tension_shift = new_values_tension
list_tension = new_list_tension


In [12]:
Model = TFTModel(list_tension, n_points)


In [13]:
gen = SyntheticFromSettings(settings, model_cls=TFTModel, out_root="synthetic_from_settings")


In [ ]:
# gerar 8 variações sintéticas por cada arquivo experimental encontrado
generated_index = gen.generate(n_variations_per_file=1, resample_n_points=10000, variation_mode="voltages", voltage_increment=6.0, voltage_max_variation=50, save_csv=True, save_npz=True)


## **Treinar modelo MLP**

In [2]:
from modules_otft._mlp_train import MLPModelTrain 

In [3]:

BASE_PATH = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/"
INDEX_PATH = f"{BASE_PATH}synthetic_from_settings/index_generated_mlpv1.json"

In [4]:
# Definição dos parâmetros do treino
HIDDEN_LAYERS = (3, 3, 3)
EPOCS = 2000  # Máximo de épocas
TEST_SPLIT = 0.2      # 20% para teste/validação
LEARNING_RATE = 0.0001
BATSIZE = 128
PATIENCE = 100


# Instanciação do Otimizador/Treinador
mlp_trainer = MLPModelTrain(
    index_path=INDEX_PATH,
    hidden_layers=HIDDEN_LAYERS,
    activation='tanh',
    learning_rate=LEARNING_RATE,
    max_iter=EPOCS,
    batch_size=BATSIZE,
    n_iter=PATIENCE,
    test_size=TEST_SPLIT,
    model_path='models/Mlp_v1/mlp_model_v1.pkl',
    load_model=False
)

print(f"Estrutura da MLP definida: Input (5 features) -> {HIDDEN_LAYERS[0]} -> {HIDDEN_LAYERS[1]} -> Output (ln|Id|)")

Estrutura da MLP definida: Input (5 features) -> 3 -> 3 -> Output (ln|Id|)


In [5]:
# Inicia o processo de treinamento
# Este método retorna o modelo treinado e o scaler_X ajustado
mlp_trainer.train_model()

Carregando índice de dados em: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/index_generated_mlpv1.json
Padronizando Features (X) e Target (Y)...
Iniciando treinamento da MLP...
Iteration 1, loss = 0.56907870
Iteration 2, loss = 0.47829393
Iteration 3, loss = 0.34495854
Iteration 4, loss = 0.14071266
Iteration 5, loss = 0.07634516
Iteration 6, loss = 0.05201022
Iteration 7, loss = 0.03546595
Iteration 8, loss = 0.02317425
Iteration 9, loss = 0.01466847
Iteration 10, loss = 0.00942149
Iteration 11, loss = 0.00644840
Iteration 12, loss = 0.00472031
Iteration 13, loss = 0.00361861
Iteration 14, loss = 0.00290123
Iteration 15, loss = 0.00243576
Iteration 16, loss = 0.00212602
Iteration 17, loss = 0.00191043
Iteration 18, loss = 0.00174906
Iteration 19, loss = 0.00161685
Iteration 20, loss = 0.00149936
Iteration 21, loss = 0.00139341
Iteration 22, loss = 0.00129552
Iteration 23, loss = 0.00120731
Iteration 24, loss = 0.00112843
Iteration 25, loss = 0.00105738


In [6]:
# Carregar o modelo treinado para inferência
trained_model, feature_scaler, target_scaler = mlp_trainer.load_model()

Modelo e Scalers carregados de models/Mlp_v1/mlp_model_v1*


In [7]:
# Importações necessárias 
from modules_otft._inferency import plot_curve_comparison, prepare_mlp_features, load_data_for_inference
import numpy as np


In [8]:
# Arquivo CSV de teste e seus metadados
TEST_CSV_PATH = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/infer_curve.json'
V_real, I_real, v_fixed, is_transfer = load_data_for_inference(TEST_CSV_PATH)



In [10]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real*1e6,  
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log",  # Escala linear para corrente
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


In [13]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real*1e6,  
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log",  # Escala linear para corrente
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)

In [9]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real*1e6,  
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log",  # Escala linear para corrente
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)